# Stage 09 — Collect the fresh research sample

One model; research question: does human-rated puzzle difficulty predict assisted exact first-move success? This separately frozen 60-puzzle sample excludes all 222 previously exposed/reserved puzzle IDs. It is an early balanced sample, not a final population study. Eighty percent or any other accuracy is not assumed.

Stage 08 must pass and be audited. The selected effort and exact prompt/settings are frozen before collection. Quota deferrals resume when Run All is repeated; terminal outcomes never get replaced. API errors, length limits, formatting errors, illegal moves and incorrect legal moves are separate. Do not publish partial collection accuracy as the planned sample result. Stage 10 fits only after the prespecified analysis gate passes.

Final answers may be one legal UCI move or one canonical legal SAN move. SAN is normalized to UCI before exact official-move scoring; no moves are extracted from prose. This rule is frozen before any fresh calls.

## Setup and immutable collection protocol

In [1]:

from pathlib import Path
from datetime import datetime,timezone
from collections import Counter
import json,hashlib,re,time,uuid,math
import requests,chess,nbformat
import pandas as pd
import numpy as np
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'experiments/stage_index.json').exists())
STAGE_NAME='09_research_sample'
CFG_DIR=ROOT/'experiments'/STAGE_NAME
CONFIG=json.loads((CFG_DIR/'config.json').read_text())
PROMPT=(CFG_DIR/'prompt.txt').read_text()
OUT=ROOT/'data/experiments'/STAGE_NAME;OUT.mkdir(parents=True,exist_ok=True)
def sha(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()
def save(name,value):
    (OUT/name).write_text(json.dumps(value,indent=2,ensure_ascii=False)+'\n',encoding='utf-8')
def records(name):
    p=OUT/name
    return [json.loads(s) for s in p.read_text(encoding='utf-8').splitlines() if s.strip()] if p.exists() else []
def append(name,row):
    with (OUT/name).open('a',encoding='utf-8') as f:f.write(json.dumps(row,ensure_ascii=False)+'\n');f.flush()
def freeze():
    n=nbformat.read(ROOT/'notebooks'/'09_collect_research_sample.ipynb',as_version=4)
    manifest={'config':CONFIG,'prompt_sha256':sha(CFG_DIR/'prompt.txt'),'notebook_code_sha256':hashlib.sha256('\n\n'.join(c.source for c in n.cells if c.cell_type=='code').encode()).hexdigest()}
    if (OUT/'manifest.json').exists():assert manifest==json.loads((OUT/'manifest.json').read_text()),'Protocol changed. Create a new version.'
    else:save('manifest.json',manifest)
freeze()
session=None
def connect():
    global session
    key=(ROOT/'api_key.txt').read_text(encoding='utf-8-sig').strip()
    assert key.startswith('gsk_'), 'Expected a locally saved Groq key'
    session=requests.Session();session.headers['Authorization']='Bearer '+key
    del key
    r=session.get(CONFIG['base_url']+'/models',timeout=30)
    if not r.ok:raise RuntimeError(f'Groq model access failed: HTTP {r.status_code}')
    assert CONFIG['model'] in {m['id'] for m in r.json()['data']}
    save('provider_models.json',r.json())
def piece_list(board):
    return '\n'.join(('White' if color else 'Black')+' pieces: '+', '.join(
        f'{chess.piece_name(p.piece_type)} {chess.square_name(s)}' for s,p in sorted(board.piece_map().items()) if p.color==color)
        for color in [chess.WHITE,chess.BLACK])
def body_for(board,task,effort):
    choices=sorted(m.uci() for m in board.legal_moves)
    prompt=PROMPT.format(side='White' if board.turn else 'Black',fen=board.fen(),pieces=piece_list(board),choices=', '.join(choices))
    if task=='side':prompt=f'Which side moves next in this FEN? {board.fen()}\n'+piece_list(board)+'\nReturn only white or black in the final answer.'
    elif task=='mate':prompt=prompt.replace('Choose the best next move','Choose a move that delivers checkmate immediately')
    elif task=='legal':prompt=prompt.replace('Choose the best next move','Choose any legal next move')
    return {'model':CONFIG['model'],'temperature':CONFIG['temperature'],'seed':CONFIG['seed'],
            'reasoning_effort':effort,'max_completion_tokens':CONFIG['max_completion_tokens'],
            'messages':[{'role':'user','content':prompt}]}
def score(board,task,response,expected=None):
    choice=response['choices'][0]
    if choice.get('finish_reason')=='length':return 'generation_limit',None,False
    if choice.get('finish_reason')!='stop':return 'nonstop_termination',None,False
    answer=choice.get('message',{}).get('content')
    if not isinstance(answer,str):return 'unparseable',None,False
    answer=answer.strip()
    if task=='side':
        if answer not in ['white','black']:return 'unparseable',answer,False
        return ('success' if answer==('white' if board.turn else 'black') else 'incorrect_side'),answer,True
    if re.fullmatch('[a-h][1-8][a-h][1-8][qrbn]?',answer):
        move=chess.Move.from_uci(answer)
        if move not in board.legal_moves:return 'illegal_move',answer,False
    else:
        try:
            move=board.parse_san(answer)
            if move not in board.legal_moves:return 'illegal_move',answer,False
            canonical=board.san(move)
            normalized=answer.replace('0-0-0','O-O-O').replace('0-0','O-O')
            if normalized not in {canonical,canonical.rstrip('+#')}:return 'unparseable',answer,False
        except ValueError:return 'unparseable',answer,False
    answer=move.uci()
    if task=='mate':
        copy=board.copy();copy.push(move)
        correct=copy.is_checkmate()
    else:correct=task=='legal' or answer==expected
    return ('success' if correct else 'legal_incorrect'),answer,True
def seconds(value):
    try:return float(value)
    except (ValueError,TypeError):
        pairs=re.findall(r'([0-9.]+)(ms|s|m|h)',str(value))
        if pairs:return sum(float(n)*{'ms':.001,'s':1,'m':60,'h':3600}[unit] for n,unit in pairs)
        try:return max(0,(datetime.strptime(value,'%a, %d %b %Y %H:%M:%S GMT').replace(tzinfo=timezone.utc)-datetime.now(timezone.utc)).total_seconds())
        except (ValueError,TypeError):return None
def pacing_wait(body):
    previous=records('requests.jsonl')
    if not previous:return 0
    row=previous[-1];headers=row.get('rate_limits',{})
    elapsed=max(0,time.time()-row.get('finished_epoch',0))
    # Conservative request reservation includes prompt estimate and the full output budget.
    need=body['max_completion_tokens']+math.ceil(sum(len(m['content']) for m in body['messages'])/2.5)+128
    limit=float(headers.get('x-ratelimit-limit-tokens',8000))
    if need>limit:raise RuntimeError('Request reservation exceeds token/minute allowance; change protocol version before collection.')
    remaining=float(headers.get('x-ratelimit-remaining-tokens',-1))
    reset=seconds(headers.get('x-ratelimit-reset-tokens'))
    if remaining<need:return max(0,(reset if reset is not None else 60)-elapsed+2)
    return max(0,2-elapsed)
def latest():return {r['case_id']:r for r in records('outcomes.jsonl')}
def run_case(case_id,puzzle_id,board,task,effort,expected=None):
    previous=latest().get(case_id)
    if previous and previous['status']!='deferred_quota':return previous
    body=body_for(board,task,effort);trial=str(uuid.uuid4());started=time.perf_counter()
    response=None;state='provider_error';last_http=None
    for retry in range(5):
        last_http=None
        wait=pacing_wait(body)
        if wait>120:state='deferred_quota';break
        if wait>0:
            print(case_id,'waiting',round(wait,1),'seconds for token capacity',flush=True);time.sleep(wait)
        log={'trial_id':trial,'case_id':case_id,'puzzle_id':puzzle_id,'task':task,'effort':effort,'retry':retry,
             'timestamp_utc':datetime.now(timezone.utc).isoformat(),'request':body}
        server_wait=None
        try:
            r=session.post(CONFIG['base_url']+'/chat/completions',json=body,timeout=(15,240))
            last_http=r.status_code;log['http_status']=r.status_code
            log['rate_limits']={k.lower():v for k,v in r.headers.items() if k.lower().startswith(('x-ratelimit','retry-after'))}
            if r.ok:
                value=r.json()
                assert isinstance(value.get('choices'),list) and value['choices']
                response=value;log['response']=value;state='completed'
            else:
                try:log['provider_error']=r.json().get('error',{})
                except ValueError:log['provider_error']={'code':'non_json_error'}
                if r.status_code==429:
                    server_wait=seconds(r.headers.get('retry-after'))
                    # Do not truncate a long Retry-After and burn retries against a daily limit.
                    if server_wait is None:server_wait=seconds(r.headers.get('x-ratelimit-reset-tokens')) or 60
                    state='deferred_quota'
        except (requests.RequestException,ValueError,AssertionError) as error:
            log['error_type']=type(error).__name__;state='transport_error'
        log['finished_epoch']=time.time();append('requests.jsonl',log)
        if response is not None:break
        if last_http==429:
            if server_wait>120:break
            print(case_id,'server requested retry after',round(server_wait,1),'seconds',flush=True)
            time.sleep(server_wait+2)
        elif last_http in [500,502,503,504,None]:time.sleep(min(30,5*2**retry))
        else:break
    category,answer,usable=score(board,task,response,expected) if response is not None else (state,None,False)
    row={'case_id':case_id,'trial_id':trial,'puzzle_id':puzzle_id,'task':task,'effort':effort,'status':state,
         'failure_type':category,'answer':answer,'usable_answer':bool(usable),
         'success':int(category=='success') if response is not None else None,
         'elapsed_seconds':round(time.perf_counter()-started,3),'http_status':last_http,
         'finish_reason':response['choices'][0].get('finish_reason') if response else None,
         'completion_tokens':response.get('usage',{}).get('completion_tokens') if response else None}
    append('outcomes.jsonl',row);print(case_id,category,row['elapsed_seconds'],'seconds',flush=True)
    return row


## Audited readiness gate

In [2]:

gate_dir=ROOT/'data/experiments/08_hosted_reliability'
gate_path=gate_dir/'readiness_gate.json';audit_path=gate_dir/'audit.json'
gate=json.loads(gate_path.read_text()) if gate_path.exists() else {}
audit=json.loads(audit_path.read_text()) if audit_path.exists() else {}
ready=bool(gate.get('ready_for_fresh_collection') and audit.get('passed'))
if ready:
    selected_effort=gate['selected_effort']
    deployment={'model':CONFIG['model'],'effort':selected_effort,'readiness_gate_sha256':sha(gate_path),'stage08_manifest_sha256':sha(gate_dir/'manifest.json')}
    if (OUT/'deployment.json').exists():assert deployment==json.loads((OUT/'deployment.json').read_text()),'Selected deployment changed; create a new experiment version.'
    else:save('deployment.json',deployment)
    connect()
    print('Audited stage-08 gate passed; selected effort:',selected_effort)
else:print('Fresh collection is blocked until stage 08 passes and its audit completes. No research API calls will be made.')


Audited stage-08 gate passed; selected effort: low


## Freeze the fresh balanced sample

In [3]:

puzzles=None
if ready:
    path=OUT/'puzzles.csv'
    if path.exists():
        puzzles=pd.read_csv(path,dtype={'puzzle_id':str})
        assert sha(path)==json.loads((OUT/'sampling.json').read_text())['sample_sha256']
    else:
        excluded=set()
        for source in [ROOT/'data/samples/pilot_v1/pilot_puzzles.csv',
                       ROOT/'data/experiments/04_local_prompt_revision/evaluation_puzzles.csv',
                       ROOT/'data/experiments/05_local_first_move/puzzles.csv',
                       ROOT/'data/experiments/07_hosted_first_move/evaluation_puzzles.csv']:
            excluded.update(pd.read_csv(source,dtype={'puzzle_id':str}).puzzle_id)
        assert len(excluded)==222
        rng=np.random.default_rng(CONFIG['sample_seed'])
        picks={(r,l):pd.DataFrame() for r in ['low','middle','high'] for l in ['one_move','multi_move']}
        feature_path=ROOT/'data/validation/full/validated_features.csv'
        source_path=ROOT/'data/lichess_db_puzzle.csv.zst'
        for chunk in pd.read_csv(feature_path,chunksize=100_000,dtype={'puzzle_id':str}):
            frame=chunk.loc[~chunk.puzzle_id.isin(excluded)].copy()
            frame['rating_stratum']=np.where(frame.puzzle_rating<=1157,'low',np.where(frame.puzzle_rating<=1700,'middle','high'))
            frame['length_group']=np.where(frame.total_solver_moves.eq(1),'one_move','multi_move')
            frame['_priority']=rng.random(len(frame))
            for key in picks:
                group=frame.loc[frame.rating_stratum.eq(key[0])&frame.length_group.eq(key[1])]
                picks[key]=pd.concat([picks[key],group.nsmallest(CONFIG['per_cell'],'_priority')],ignore_index=True).nsmallest(CONFIG['per_cell'],'_priority')
        puzzles=pd.concat(picks.values(),ignore_index=True).drop(columns='_priority')
        wanted=set(puzzles.puzzle_id);assert len(wanted)==CONFIG['sample_n'] and not wanted&excluded
        found=[]
        for chunk in pd.read_csv(source_path,compression='zstd',chunksize=100_000,dtype=str,keep_default_na=False,usecols=['PuzzleId','FEN','Moves','GameUrl']):
            matched=chunk.loc[chunk.PuzzleId.isin(wanted)]
            if not matched.empty:found.append(matched)
        source=pd.concat(found).rename(columns={'PuzzleId':'puzzle_id','FEN':'original_fen','Moves':'official_moves','GameUrl':'game_url'})
        puzzles=puzzles.merge(source,on='puzzle_id',validate='one_to_one').sort_values('puzzle_id')
        puzzles['official_first_move']=puzzles.official_moves.str.split().str[1]
        puzzles['random_success_probability']=1/puzzles.legal_moves
        for row in puzzles.itertuples(index=False):
            board=chess.Board(row.original_fen)
            for i,text in enumerate(row.official_moves.split()):
                assert chess.Move.from_uci(text) in board.legal_moves
                board.push_uci(text)
                if i==0:assert board.fen()==row.solver_fen and board.legal_moves.count()==row.legal_moves
        puzzles.to_csv(path,index=False)
        save('sampling.json',{'sample_sha256':sha(path),'n':len(puzzles),'seed':CONFIG['sample_seed'],
                             'excluded_ids':sorted(excluded),'features_sha256':sha(feature_path),'source_sha256':sha(source_path)})
    display(puzzles.groupby(['rating_stratum','length_group']).size().rename('puzzles'))


rating_stratum  length_group
high            multi_move      10
                one_move        10
low             multi_move      10
                one_move        10
middle          multi_move      10
                one_move        10
Name: puzzles, dtype: int64

## Collect, resume, summarize and audit

In [4]:

results=[];paused=False
if ready:
    for row in puzzles.itertuples(index=False):
        result=run_case('research:'+row.puzzle_id,row.puzzle_id,chess.Board(row.solver_fen),'evaluation',selected_effort,row.official_first_move)
        results.append(result)
        if result['status']=='deferred_quota':paused=True;break
    available={r['puzzle_id']:r for r in latest().values() if r['case_id'].startswith('research:')}
    result_frame=pd.DataFrame(available.values())
    combined=puzzles.merge(result_frame,on='puzzle_id',how='left',validate='one_to_one')
    combined.to_csv(OUT/'results.csv',index=False)
    done=combined.status.eq('completed');positive=combined.success.eq(1)&done;negative=combined.success.eq(0)&done
    all_attempted=combined.case_id.notna().all() and not combined.status.eq('deferred_quota').any()
    analysis_ready=bool(all_attempted and done.sum()>=58 and positive.sum()>=15 and negative.sum()>=15)
    summary={'planned':len(puzzles),'cases_with_outcomes':len(available),'completed_provider_responses':int(done.sum()),
             'correct':int(positive.sum()),'incorrect_or_unusable_completed':int(negative.sum()),
             'usable_moves':int(combined.usable_answer.eq(True).sum()),'paused_for_quota':paused,
             'accuracy_completed_responses':float(positive.sum()/done.sum()) if done.sum() else None,
             'fraction_of_planned_with_correct_answer':float(positive.sum()/len(puzzles)),
             'ready_for_analysis':analysis_ready,
             'note':'Incomplete collection is not an accuracy estimate for the planned sample. Provider errors are reported separately, not treated as observed incorrect chess decisions.'}
    save('collection_summary.json',summary)
    # Audit raw request and response scoring, including every terminal error.
    responses={r['trial_id']:r for r in records('requests.jsonl') if 'response' in r}
    by_id={r.puzzle_id:r for r in puzzles.itertuples(index=False)};verified=0
    for result in latest().values():
        if result['status']!='completed':continue
        row=by_id[result['puzzle_id']];board=chess.Board(row.solver_fen);log=responses[result['trial_id']]
        assert log['request']==body_for(board,'evaluation',selected_effort)
        category,answer,usable=score(board,'evaluation',log['response'],row.official_first_move)
        assert result['failure_type']==category and result['answer']==answer and result['success']==int(category=='success') and result['usable_answer']==usable
        verified+=1
    assert sha(OUT/'puzzles.csv')==json.loads((OUT/'sampling.json').read_text())['sample_sha256']
    save('audit.json',{'passed':True,'responses_verified':verified,'sample_sha256':sha(OUT/'puzzles.csv'),'results_sha256':sha(OUT/'results.csv')})
    session.close();print(json.dumps(summary,indent=2));display(combined.groupby('rating_stratum').agg(planned=('puzzle_id','size'),returned=('success','count'),correct=('success','sum')))
else:print('Stage 09 not started; retain the stage-08 diagnostic outcome and resolve readiness first.')


research:0kWbW legal_incorrect 3.168 seconds


research:17t2T waiting 16.8 seconds for token capacity


research:17t2T success 20.79 seconds


research:2rXfR waiting 15.1 seconds for token capacity


research:2rXfR success 16.257 seconds


...

research:wpm0h legal_incorrect 5.74 seconds


research:yq7rb waiting 14.9 seconds for token capacity


research:yq7rb success 17.399 seconds


{
  "planned": 60,
  "cases_with_outcomes": 60,
  "completed_provider_responses": 60,
  "correct": 21,
  "incorrect_or_unusable_completed": 39,
  "usable_moves": 59,
  "paused_for_quota": false,
  "accuracy_completed_responses": 0.35,
  "fraction_of_planned_with_correct_answer": 0.35,
  "ready_for_analysis": true,
  "note": "Incomplete collection is not an accuracy estimate for the planned sample. Provider errors are reported separately, not treated as observed incorrect chess decisions."
}


,planned,returned,correct
rating_stratum,,,
high,20,20,6
low,20,20,9
middle,20,20,6
